In [1]:
%matplotlib notebook

In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
import numpy as onp
from tqdm import tqdm
import matplotlib.pyplot as plt

from msmjax.kernels import split_one_over_r_kernel, SofteningFunctionOneOverR
from msmjax.shortrange import make_compute_U_zero_with_neighborlist, make_compute_f_zero_with_neighborlist, make_compute_U_and_f_zero_with_neighborlist
from msmjax.gridops_multidim import set_up_grids_all_levels
from msmjax.gridops_multidim import create_compute_U_oneplus, create_compute_U_and_f_oneplus


# Not working

In [3]:
# AVG_NEIGHBOR_DISTANCE = 2.5
# 
# N_DIM = 3
# PBCS = [False] * N_DIM
# 
# N_LEVELS = 4
# ALPHA = 3
# P = 6
# MU = 3
# LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
# LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

In [4]:
# n_points_per_direction = 15
# n_particles = n_points_per_direction**N_DIM
# 
# rng = onp.random.default_rng(2489)
# 
# # # Randomly drawing positions causes errors during neighbor list allocation.
# # # (I assume some problem due to particles being too close together, although I don't see why it would matter)
# # side_length = n_particles ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
# # box_lengths = jnp.array([side_length] * N_DIM)
# # pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(n_particles, N_DIM))
# 
# # For now, place particles on a regular grid instead
# side_length = (n_points_per_direction + 1) * AVG_NEIGHBOR_DISTANCE
# box_lengths = jnp.array([side_length] * N_DIM)
# mg = onp.meshgrid(*([onp.arange(n_points_per_direction)] * N_DIM), indexing="ij")
# pos = onp.stack([m.ravel() for m in mg], axis=1) * AVG_NEIGHBOR_DISTANCE
# 
# chg = rng.uniform(low=-1.0, high=1.0, size=n_particles)
# 
# pos = jnp.array(pos)
# chg = jnp.array(chg)

In [5]:
# import matplotlib.pyplot as plt
# 
# try:
#     x, y, z, = pos[:, 0], pos[:, 1], pos[:, 2]
# 
#     fig = plt.figure()
#     ax = fig.add_subplot(projection="3d")
#     ax.scatter(x.ravel(), y.ravel(), z.ravel(), s=10)
# 
#     plt.show()
# except ValueError:
#     pass

In [6]:
# kernels = split_one_over_r_kernel(
#         max_level=N_LEVELS,
#         level_zero_cutoff=LEVEL_ZERO_CUTOFF,
#         softening_function=SofteningFunctionOneOverR(P),
#     )

In [7]:
# neighbor_fun, compute_U_zero_with_neighborlist = make_compute_U_zero_with_neighborlist(
#     kernels=kernels,
#     cutoff=LEVEL_ZERO_CUTOFF,
#     box_lengths=box_lengths,
#     pbcs=PBCS,
# )
# nbl_allocate_fun = neighbor_fun.allocate
# nbl_update_fun = neighbor_fun.update
# neighbor_list = nbl_allocate_fun(pos)


In [8]:
# from jax_md import space, partition
# 
# if PBCS[0]:
#     displacement_fn, shift_fn = space.periodic(box_lengths)
# else:
#     displacement_fn, shift_fn = space.free()
# neighbor_fn = partition.neighbor_list(
#     displacement_fn, box_lengths, r_cutoff=LEVEL_ZERO_CUTOFF
# )
# neighborlist = neighbor_fn.allocate(pos)

In [9]:
# @jax.jit
# def wrapped_compute_U_zero_with_neighborlist(positions, charges):
#     updated_neighbor_list = neighbor_fun.update(positions, neighbor_list)
#     return compute_U_zero_with_neighborlist(positions, charges, updated_neighbor_list.idx)

In [10]:
# wrapped_compute_U_zero_with_neighborlist(pos, chg)

# Working code copied from `shortrange.py`

In [11]:
# # Structure settings
# # BOX_LENGTHS = jnp.array([10.0, 12.0, 17.5, 20.0])
# BOX_LENGTHS = jnp.array([10.0, 12.0, 17.5])
# # BOX_LENGTHS = jnp.array([10.0, 12.0])
# # BOX_LENGTHS = jnp.array([10.0])
# PERIODIC = True
# N_PARTICLES = 50
# 
# # MSM settings
# LEVEL_ZERO_CUTOFF = 3.5
# MAX_GRIDLEVEL = 4
# P = 4
# 
# n_dim = len(BOX_LENGTHS)
# pbcs = [PERIODIC] * n_dim
# 
# rng = onp.random.default_rng(58347)
# pos = rng.uniform(
#     low=[0.0] * n_dim, high=BOX_LENGTHS, size=(N_PARTICLES, n_dim)
# )
# chg = rng.uniform(low=-1.0, high=1.0, size=N_PARTICLES)
# pos = jnp.array(pos)
# chg = jnp.array(chg)
# 
# kernels = split_one_over_r_kernel(
#     max_level=MAX_GRIDLEVEL,
#     level_zero_cutoff=LEVEL_ZERO_CUTOFF,
#     softening_function=SofteningFunctionOneOverR(P),
# )
# 
# neighbor_fun, energy_fun = make_compute_U_zero_with_neighborlist(
#     kernels=kernels,
#     cutoff=LEVEL_ZERO_CUTOFF,
#     box_lengths=BOX_LENGTHS,
#     pbcs=pbcs,
# )
# nbl_allocate_fun = neighbor_fun.allocate
# nbl_update_fun = neighbor_fun.update
# energy_fun = jax.jit(energy_fun)
# neighborlist = nbl_allocate_fun(pos)
# e_neighborlist = energy_fun(pos, chg, neighborlist.idx)
# print(e_neighborlist)

# Investigate

In [12]:
# AVG_NEIGHBOR_DISTANCE = 2.5
# 
# N_PARTICLES = 700
# N_DIM = 3
# PBCS = [False] * N_DIM
# 
# side_length = N_PARTICLES ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
# box_lengths = jnp.array([side_length] * N_DIM)
# N_LEVELS = 4
# ALPHA = 3
# P = 6
# MU = 3
# LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
# LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

In [13]:
# kernels = split_one_over_r_kernel(
#         max_level=N_LEVELS,
#         level_zero_cutoff=LEVEL_ZERO_CUTOFF,
#         softening_function=SofteningFunctionOneOverR(P),
#     )

## Random positions

In [14]:
# # rng = onp.random.default_rng(2489)
# rng = onp.random.default_rng(4237)
# 
# pos = rng.uniform(low=onp.zeros_like(box_lengths), high=box_lengths, size=(N_PARTICLES, N_DIM))
# chg = rng.uniform(low=-1.0, high=1.0, size=N_PARTICLES)
# pos = jnp.array(pos)
# chg = jnp.array(chg)

In [15]:
# for _ in tqdm(range(100)):
#     neighbor_fun, energy_fun = make_compute_U_zero_with_neighborlist(
#         kernels=kernels,
#         cutoff=LEVEL_ZERO_CUTOFF,
#         # box_lengths=BOX_LENGTHS,
#         # pbcs=pbcs,
#         box_lengths=box_lengths,
#         pbcs=PBCS,
#     )
#     nbl_allocate_fun = neighbor_fun.allocate
#     nbl_update_fun = neighbor_fun.update
#     energy_fun = jax.jit(energy_fun)
#     neighborlist = nbl_allocate_fun(pos)
#     e_neighborlist = energy_fun(pos, chg, neighborlist.idx)
#     # print(e_neighborlist)

## Positions on grid

In [16]:
# (box_lengths / AVG_NEIGHBOR_DISTANCE).astype(int)

In [17]:
# onp.linspace(0, box_lengths, int(box_lengths[0] / AVG_NEIGHBOR_DISTANCE) + 1)

In [18]:
# # n_points_per_direction = int(box_lengths[0] / AVG_NEIGHBOR_DISTANCE) + 1
# n_points_per_direction = 15
# mg = onp.meshgrid(*([onp.arange(n_points_per_direction)] * N_DIM), indexing="ij")
# pos_grid = onp.stack([m.ravel() for m in mg], axis=1) * AVG_NEIGHBOR_DISTANCE
# chg_grid = onp.ones(pos_grid.shape[0], dtype=float)

In [19]:
# pos_grid.shape

In [20]:
# for _ in tqdm(range(25)):
#     neighbor_fun, energy_fun = make_compute_U_zero_with_neighborlist(
#         kernels=kernels,
#         cutoff=LEVEL_ZERO_CUTOFF,
#         # box_lengths=BOX_LENGTHS,
#         # pbcs=pbcs,
#         box_lengths=box_lengths,
#         pbcs=PBCS,
#     )
#     nbl_allocate_fun = neighbor_fun.allocate
#     nbl_update_fun = neighbor_fun.update
#     energy_fun = jax.jit(energy_fun)
#     neighborlist = nbl_allocate_fun(pos_grid)
#     e_neighborlist = energy_fun(pos_grid, chg_grid, neighborlist.idx)
#     # print(e_neighborlist)

In [21]:
# e_neighborlist

In [22]:
# try:
#     x, y, z, = pos_grid[:, 0], pos_grid[:, 1], pos_grid[:, 2]
#     
#     fig = plt.figure()
#     ax = fig.add_subplot(projection="3d")
#     ax.scatter(x.ravel(), y.ravel(), z.ravel(), s=10)
#     
#     plt.show()
# except ValueError:
#     pass

# Minimal

In [23]:
AVG_NEIGHBOR_DISTANCE = 2.5

N_PARTICLES = 700
N_DIM = 3
PBCS = [False] * N_DIM

side_length = N_PARTICLES ** (1. / N_DIM) * AVG_NEIGHBOR_DISTANCE
box_lengths = jnp.array([side_length] * N_DIM)
N_LEVELS = 4
ALPHA = 3
P = 6
MU = 3
LEVEL_ONE_GRIDSPACING = AVG_NEIGHBOR_DISTANCE
LEVEL_ZERO_CUTOFF = ALPHA * LEVEL_ONE_GRIDSPACING

In [24]:
kernels = split_one_over_r_kernel(
        max_level=N_LEVELS,
        level_zero_cutoff=LEVEL_ZERO_CUTOFF,
        softening_function=SofteningFunctionOneOverR(P),
    )

In [25]:
# n_points_per_direction = int(box_lengths[0] / AVG_NEIGHBOR_DISTANCE) + 1
n_points_per_direction = 15
mg = onp.meshgrid(*([onp.arange(n_points_per_direction)] * N_DIM), indexing="ij")
pos_grid = onp.stack([m.ravel() for m in mg], axis=1) * AVG_NEIGHBOR_DISTANCE
chg_grid = onp.ones(pos_grid.shape[0], dtype=float)

In [26]:
for _ in tqdm(range(25)):
    neighbor_fun, energy_fun = make_compute_U_zero_with_neighborlist(
        kernels=kernels,
        cutoff=LEVEL_ZERO_CUTOFF,
        # box_lengths=BOX_LENGTHS,
        # pbcs=pbcs,
        box_lengths=box_lengths,
        pbcs=PBCS,
    )
    nbl_allocate_fun = neighbor_fun.allocate
    nbl_update_fun = neighbor_fun.update
    energy_fun = jax.jit(energy_fun)
    neighborlist = nbl_allocate_fun(pos_grid)
    e_neighborlist = energy_fun(pos_grid, chg_grid, neighborlist.idx)
    # print(e_neighborlist)

  0%|          | 0/25 [00:00<?, ?it/s]/home/florian/anaconda3/envs/msmjax-dev/lib/python3.8/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=int32. In future JAX releases this will result in an error.
  warnings.warn("scatter inputs have incompatible types: cannot safely cast "
100%|██████████| 25/25 [00:10<00:00,  2.39it/s]
